# Face 07 · Enrollment Quality Gates + Templates

Implements the enrollment rules from spec section 11.2 – the same thresholds go into the Android/iOS code:
exactly **one face**, **eye distance > 80 px**, sharp enough, well exposed, **pose < 20°**, liveness passed on each
frame, then **average 3–5 embeddings** into the worker's template (stored encrypted with the tenant key).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import cv2, numpy as np
from Common_Utils.Data_Utils import variance_of_laplacian

GATES = {"min_eye_distance_px": 80, "min_sharpness": 60.0, "brightness_range": (60, 200), "max_yaw_deg": 20, "max_pitch_deg": 20}

def estimate_yaw_pitch(landmarks):
    """Rough pose from the 5 YuNet points (eyes, nose, mouth corners)."""
    le, re, nose, lm, rm = landmarks
    eye_mid, mouth_mid = (le + re) / 2, (lm + rm) / 2
    eye_dist = np.linalg.norm(re - le)
    yaw = np.degrees(np.arctan2((nose[0] - eye_mid[0]), eye_dist)) * 2
    pitch = np.degrees(np.arctan2((nose[1] - (eye_mid[1] + mouth_mid[1]) / 2), eye_dist)) * 2
    return float(yaw), float(pitch)

def check_frame(image_bgr, faces):
    if len(faces) != 1:
        return False, "one_face"
    lm = faces[0][4:14].reshape(5, 2)
    if np.linalg.norm(lm[1] - lm[0]) < GATES["min_eye_distance_px"]:
        return False, "too_far"
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    if variance_of_laplacian(gray) < GATES["min_sharpness"]:
        return False, "blurry"
    lo, hi = GATES["brightness_range"]
    if not lo <= gray.mean() <= hi:
        return False, "exposure"
    yaw, pitch = estimate_yaw_pitch(lm)
    if abs(yaw) > GATES["max_yaw_deg"] or abs(pitch) > GATES["max_pitch_deg"]:
        return False, "pose"
    return True, "ok"

In [ ]:
def make_template(embeddings):
    """Average 3-5 L2-normalised embeddings -> one template (re-normalised)."""
    e = np.stack([x / np.linalg.norm(x) for x in embeddings])
    assert 3 <= len(e) <= 5, "Enrollment needs 3-5 good frames"
    t = e.mean(0)
    return t / np.linalg.norm(t), float(np.min(e @ t))    # template + consistency score (quality_score)

template, quality = make_template([np.random.randn(512) for _ in range(4)])
template.shape, round(quality, 3)